# ABIDE HeteroGNN on Colab (T4 GPU)

Runs the nested-LOSO experiments on a Colab GPU. The only input not in the
GitHub repo is `features/abide_features_raw.csv` (gitignored, subject-level, ~4.5 MB) —
you upload that one file in step 4.

**Set the runtime first:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L

In [ ]:
# 2. Get the code
!git clone https://github.com/hazrotali2003028/Thesis-with-ABIDE-dataset.git
%cd Thesis-with-ABIDE-dataset

In [ ]:
# 3. Dependencies (torch + CUDA are preinstalled on Colab)
!pip -q install torch_geometric
# If you later hit a torch-scatter/torch-sparse error, uncomment:
# import torch; v = torch.__version__
# !pip -q install pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{v}.html

In [ ]:
# 4. Upload the one gitignored input into features/
from google.colab import files
up = files.upload()            # choose abide_features_raw.csv
!mv abide_features_raw.csv features/
!ls -la features/abide_features_raw.csv

In [ ]:
# 5. (recommended) Mount Drive so results survive a disconnect.
# The scripts are resumable: rerun and they skip finished (site, seed) rows.
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

## Paper-1-rule adaptive run (what you asked for)

Adaptive model, 5 adaptive seeds, per-site HP reused from the committed
`results/nested_loso_adaptive_hp.json`. Only the epoch-selection rule changes.
Smoke-test first, then the full run. `--epochs 60` matches the group-level
paper-1 check so the two models are on the same footing.

In [ ]:
!python train_paper1rule_adaptive.py --smoke

In [ ]:
# Full run. Re-run this cell after a disconnect; it resumes.
!python train_paper1rule_adaptive.py --epochs 60
# keep a copy of results on Drive
!cp results/paper1rule_adaptive_results.csv /content/drive/MyDrive/thesis_results/ 2>/dev/null || true

## (optional) Re-run the honest adaptive nested-LOSO from scratch

This is the ~21 h job (inner HP search + outer fits). Only needed if you want to
reproduce `nested_loso_adaptive_results.csv` on the GPU; otherwise skip it.

In [ ]:
# !python train_nested_loso_adaptive.py --smoke
# !python train_nested_loso_adaptive.py